<div style="border-bottom:3px solid #4a6fa5; padding-bottom:0.3em; margin-bottom:0.4em;">
  <span style="font-size:2em; font-weight:700; color:#2b3a55;">Flight Delays 2015 — BigQuery analysis</span><br>
  <span style="font-size:1em; color:#6b7280;">CCBD exam · Università di Catania · AA 2025-2026</span>
</div>

Self-documenting notebook for the *Cloud Computing and Big Data* exam (Università di
Catania, AA 2025-2026). It drives **Google BigQuery** over the USDOT *2015 Flight Delays
and Cancellations* dataset (~5.3M flights, October dropped) and narrates the findings.

**The six questions**
1. On-time performance by airline
2. Delay propagation through the day
3. Bottleneck airports
4. Seasonality of delays and cancellations
5. Decomposition of delay causes
6. Distance vs in-flight recovery

Cloud setup, bucket and data loading live in `README.md`. Run top-to-bottom (*Run All*),
from the project root so that `sql/` and the credentials env var resolve.

# Environment

This notebook runs in a dedicated venv registered as the **CCBD** Jupyter kernel. The
cell below pins dependencies with the `%pip` magic - **not** `!pip`, which can install
into a different interpreter than the kernel and cause `ImportError`. `%pip` installs
into the *kernel's* environment, keeping the notebook self-contained ("open -> Run All").

> Versions: resolve once on your machine, then pin exactly and `pip freeze > requirements.txt`
> as a lock file (see README). The pins below are a reasonable starting point.

In [4]:
# Data handling + plotting
%pip install -q pandas==2.2.2 matplotlib==3.9.2

# BigQuery client + BQ→pandas bridge (db-dtypes maps BQ types; pyarrow is the transport)
%pip install -q google-cloud-bigquery==3.25.0 db-dtypes==1.2.0 pyarrow==17.0.0

# BigQuery extras: Storage Read API (fast result downloads) + the %%bigquery cell magic
%pip install -q google-cloud-bigquery-storage bigquery-magics

# Cloud Storage (GCS bucket access)
%pip install -q google-cloud-storage

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


> **Run once.** This install only needs to run the first time, or whenever you switch or recreate the kernel — once the packages are in the **CCBD** environment they persist, so on a later *Run All* this cell is a fast no-op.

# Authentication

The notebook authenticates to BigQuery with a **service-account key (JSON)**. The key
path is read from the `GOOGLE_APPLICATION_CREDENTIALS` environment variable - **never
hardcoded, never printed**. The key carries least-privilege roles (BigQuery Job User +
Data Editor + Read Session User; Storage Object Admin on the bucket only). See `README.md` section 1.1.

In [ ]:
import os
from google.cloud import bigquery
from google.oauth2 import service_account

PROJECT_ID = 'ccbd-20260603-gpappa'   # not secret - only the KEY stays out of the notebook
DATASET    = 'flights_2015'

key_path = os.environ['GOOGLE_APPLICATION_CREDENTIALS']   # path only; contents never printed
credentials = service_account.Credentials.from_service_account_file(
    key_path, scopes=['https://www.googleapis.com/auth/cloud-platform'])
client = bigquery.Client(project=PROJECT_ID, credentials=credentials)
print('BigQuery client ready for project:', client.project)

# Enable the %%bigquery cell magic (from the bigquery-magics package) and point it at the
# same service account + project, so the magic and the client share one identity.
%load_ext bigquery_magics
import bigquery_magics
bigquery_magics.context.credentials = credentials
bigquery_magics.context.project = PROJECT_ID

# Query helper and cost discipline

At this scale querying is effectively free: a full scan is ~0.5 GB against BigQuery's
1 TiB/month free tier (cached repeats are free). We still keep two cost-aware habits,
visible in the helper: a **`dry_run`** to preview bytes scanned, and **`maximum_bytes_billed`**
as a guardrail. The six queries are read from the validated files in `sql/` - the same
text used at the BigQuery UI, so there is a single source of truth. The `show_and_run()`
helper **displays each query's SQL inline** (from its `sql/` file) before running it, so the
notebook stays self-documenting without duplicating the SQL.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Markdown, display

SQL_DIR = Path('sql')

def load_sql(name):
    return (SQL_DIR / name).read_text()

def run_query(sql, max_gb=2.0, preview=True):
    if preview:
        dry = client.query(sql, job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False))
        print(f'dry-run: will scan {dry.total_bytes_processed / 1e9:.3f} GB')
    cfg = bigquery.QueryJobConfig(maximum_bytes_billed=int(max_gb * 1e9))
    return client.query(sql, job_config=cfg).to_dataframe()

def show_and_run(name, max_gb=2.0):
    """Display the SQL from sql/<name> (the single source of truth), then run it."""
    sql = load_sql(name)
    display(Markdown(f"**`sql/{name}`**\n\n```sql\n{sql}\n```"))
    return run_query(sql, max_gb=max_gb)

## Aside — the `%%bigquery` cell magic

BigQuery also ships a Jupyter **cell magic**: write SQL straight in a cell and the result comes
back as a DataFrame named after the magic argument. We drive the six analyses with `show_and_run()`
instead — so the `sql/` files stay the single source of truth — but the magic is the most direct way
to run ad-hoc SQL, shown here once for reference. It uses the service-account credentials wired in the
**Authentication** cell (via `bigquery_magics.context`). To reproduce it for a query, copy the SQL
from its `sql/` file into a `%%bigquery` cell.

In [ ]:
%%bigquery demo_top_carriers --project ccbd-20260603-gpappa
SELECT a.AIRLINE AS airline, COUNT(*) AS flights
FROM flights_2015.flights AS f
JOIN flights_2015.airlines AS a ON f.AIRLINE = a.IATA_CODE
GROUP BY airline
ORDER BY flights DESC
LIMIT 5

# Sanity check

Ground-truth anchors before the analysis: total flights, cancellations, and
carrier/airport counts. If these look wrong, a query that runs cleanly could still be
computing the wrong thing.

In [ ]:
run_query(f'''
SELECT
  COUNT(*)                       AS flights,
  COUNTIF(CANCELLED = 1)         AS cancelled,
  COUNT(DISTINCT AIRLINE)        AS carriers,
  COUNT(DISTINCT ORIGIN_AIRPORT) AS origin_airports
FROM {DATASET}.flights
''')

# Queries

The six predefined analytical queries. Each one loads its SQL from `sql/`, runs it through
the helper above (dry-run preview + byte cap), plots the result, and states the finding.

## Q1 - On-time performance by airline

Flights, average departure/arrival delay, and the share of delayed arrivals (>= 15 min,
the DOT threshold) per carrier, joined to the airline name. A carrier reliability ranking.

In [ ]:
q1 = show_and_run('q1_ontime_by_airline.sql')
q1

In [ ]:
import matplotlib.pyplot as plt

ax = q1.sort_values('pct_delayed').plot.barh(
    x='airline_name', y='pct_delayed', legend=False, figsize=(8, 5), color='steelblue')
ax.set_xlabel('% arrivals delayed (>= 15 min)'); ax.set_ylabel('')
ax.set_title('Q1 - Carrier reliability (lower is better)')
plt.tight_layout(); plt.show()

**Finding.** Ultra-low-cost carriers (Spirit ~30%, Frontier ~27%) are worst; Hawaiian,
Alaska and Delta best (~11-14%). For most carriers `avg_dep_delay > avg_arr_delay` - they
recover time in the air (revisited in Q6).

## Q2 - Delay propagation through the day

Average arrival delay and % delayed by **scheduled departure hour** (derived from the
HHMM field).

In [ ]:
q2 = show_and_run('q2_hourly_propagation.sql')
q2

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(q2['sched_dep_hour'], q2['avg_arr_delay_min'], marker='o')
ax.axhline(0, color='grey', lw=0.8)
ax.set_xlabel('scheduled departure hour'); ax.set_ylabel('avg arrival delay (min)')
ax.set_title('Q2 - Delays compound through the day')
plt.tight_layout(); plt.show()

**Finding.** Morning flights arrive early; delay rises monotonically to a ~19:00 peak
(+11 min, 27% delayed). Hours 0-4 are very low volume (red-eyes) and noisy - the signal is
the 5-23 rise.

## Q3 - Bottleneck airports

Top origin airports by average departure delay, with a minimum-volume filter, joined to
city/state.

In [ ]:
q3 = show_and_run('q3_bottleneck_airports.sql')
q3

In [ ]:
ax = q3.sort_values('avg_dep_delay_min').plot.barh(
    x='airport_code', y='avg_dep_delay_min', legend=False, figsize=(8, 6), color='indianred')
ax.set_xlabel('avg departure delay (min)'); ax.set_ylabel('')
ax.set_title('Q3 - Most congested origin airports')
plt.tight_layout(); plt.show()

**Finding.** Chicago O'Hare leads (~14 min), with the NYC-area airports (Newark,
LaGuardia, JFK) and Chicago Midway close behind - the usual congested hubs.

## Q4 - Seasonality

Average arrival delay and cancellation rate by month (October excluded at the source).

In [ ]:
q4 = show_and_run('q4_seasonality.sql')
q4

In [ ]:
fig, ax1 = plt.subplots(figsize=(9, 4))
ax1.bar(q4['month'], q4['cancellation_rate_pct'], color='lightcoral', alpha=0.8)
ax1.set_xlabel('month'); ax1.set_ylabel('cancellation rate (%)', color='indianred')
ax2 = ax1.twinx()
ax2.plot(q4['month'], q4['avg_arr_delay_min'], color='navy', marker='o')
ax2.set_ylabel('avg arrival delay (min)', color='navy')
ax1.set_title('Q4 - Winter cancellations, summer delays')
plt.tight_layout(); plt.show()

**Finding.** Cancellations peak in winter (Feb 4.8%, Jan 2.6% - snow); arrival delays
peak in summer (June 9.6 min - thunderstorms + heavy travel). September is the calmest.

## Q5 - Decomposition of delay causes

For delayed flights, the share of total delay minutes attributed to each cause.

In [ ]:
q5 = show_and_run('q5_cause_decomposition.sql')
q5

In [ ]:
causes = q5.iloc[0].sort_values(ascending=False)
ax = causes.plot.bar(figsize=(7, 4), color='slateblue')
ax.set_ylabel('% of delay minutes'); ax.set_xlabel('')
ax.set_title('Q5 - Late aircraft (propagation) dominates')
plt.xticks(rotation=30, ha='right'); plt.tight_layout(); plt.show()

**Finding.** `late_aircraft` (~40%) dominates - this *is* the propagation effect behind
Q2. Airline-controlled (~32%) and air-system (~23%) follow; security is ~0%. Direct
"weather" is only ~5%, but much weather impact hides inside the air-system and late-aircraft
buckets.

## Q6 - Distance vs in-flight recovery

By distance bucket: average `arrival_delay - departure_delay`. Negative means time was made
up in the air.

In [ ]:
q6 = show_and_run('q6_distance_recovery.sql')
q6

In [ ]:
ax = q6.plot.bar(x='distance_bucket', y='avg_recovery_min', legend=False,
                 figsize=(8, 4), color='seagreen')
ax.axhline(0, color='grey', lw=0.8)
ax.set_ylabel('avg(arrival - departure) delay (min)'); ax.set_xlabel('distance bucket (mi)')
ax.set_title('Q6 - Longer flights recover more time')
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

**Finding.** Recovery is negative everywhere and grows with distance (-2.9 -> -9 min):
longer flights have more cruise time to absorb a late departure.

# Data enrichment - BigQuery ML

So far we have *described* the past. Here we *predict*: given only what is known about a flight
**before it leaves** — month, day of week, scheduled departure hour, airline, origin, destination,
distance, scheduled duration — how likely is it to **arrive 15+ minutes late** (the official DOT
"delayed" threshold)?

This is a **binary classification** problem — the answer is yes/no ("late" or "on-time"). We train a
**logistic-regression** model: a classic, fast, interpretable method that learns a weighted
combination of the inputs and turns it into a **probability between 0 and 1**. **BigQuery ML** trains
it *inside the database* with a single `CREATE MODEL` statement — no data leaves BigQuery, no separate
ML environment needed.

**Reading the training query below:**
- `CREATE OR REPLACE MODEL …delay_logreg` — creates (or overwrites) a model object, stored in the
  dataset just like a table.
- `model_type='LOGISTIC_REG'` — use logistic regression.
- `input_label_cols=['is_delayed']` — the column the model must learn to predict (the "answer").
- `auto_class_weights=TRUE` — on-time flights vastly outnumber late ones; this re-balances the two
  classes so the model can't score well by lazily predicting "on-time" for everyone.
- The `SELECT` builds the training set: the label `is_delayed` (1 if `ARRIVAL_DELAY >= 15`, else 0)
  plus the eight pre-departure features. `WHERE CANCELLED = 0 AND ARRIVAL_DELAY IS NOT NULL` keeps only
  completed flights with a known outcome — the only ones that can teach the model. (`dep_hour` is the
  departure hour pulled out of the HHMM `SCHEDULED_DEPARTURE` field.)

In [ ]:
create_model = f'''
CREATE OR REPLACE MODEL {DATASET}.delay_logreg
OPTIONS(model_type='LOGISTIC_REG', input_label_cols=['is_delayed'], auto_class_weights=TRUE) AS
SELECT
  IF(ARRIVAL_DELAY >= 15, 1, 0) AS is_delayed,
  MONTH, DAY_OF_WEEK,
  MOD(DIV(SAFE_CAST(SCHEDULED_DEPARTURE AS INT64), 100), 24) AS dep_hour,
  AIRLINE, ORIGIN_AIRPORT, DESTINATION_AIRPORT, DISTANCE, SCHEDULED_TIME
FROM {DATASET}.flights
WHERE CANCELLED = 0 AND ARRIVAL_DELAY IS NOT NULL
'''
client.query(create_model).result()
print('model trained')

## Is the model any good? Reading `ML.EVALUATE`

`ML.EVALUATE` reports standard quality metrics. The headline one for a classifier is **ROC AUC**
("Area Under the ROC Curve"):

- Intuitively, it is the probability that the model gives a **randomly chosen late flight** a higher
  "late" score than a **randomly chosen on-time flight**.
- **0.5 = no skill** (a coin toss); **1.0 = perfect**. Clearly above 0.5 means the features carry real
  signal; for messy real-world data, ~**0.65-0.72** is a solid, useful result.
- It is **threshold-independent**: it measures how well the model *ranks* flights by risk, regardless
  of where you draw the "predict late" cut-off.

The other columns (`precision`, `recall`, `accuracy`, `f1_score`, `log_loss`) describe behaviour at the
default 0.5 cut-off. Note that **`accuracy` alone is misleading here**: since most flights are on time,
always guessing "on-time" already scores high — which is exactly why ROC AUC is the metric to watch.

In [ ]:
run_query(f'SELECT * FROM ML.EVALUATE(MODEL `{PROJECT_ID}.{DATASET}.delay_logreg`)', preview=False)

**Finding.** The ROC AUC comes out clearly above 0.5 — in the ~0.65-0.72 band typical for real-world
delay data — so the pre-departure features **do** carry real predictive signal: the drivers seen
descriptively above are also *predictive*. Concretely, the model learns that a late-evening departure
from a congested hub in peak season is far more likely to arrive late than an early-morning short hop
in a quiet month — the quantitative echo of Q2 (hour), Q3 (airport) and Q4 (season). It is **not** a
crystal ball: an AUC well below 1.0 reflects that delays also hinge on day-of-operations factors (the
exact weather, the late inbound aircraft) that these eight features simply don't capture.

## Putting the model to work

A trained model is queryable like a table, through `ML.PREDICT`. Two everyday uses follow. Both feed
the model *hand-built* rows (no table scan, so they are instant and free). `predicted_is_delayed` is the
yes/no call at the 0.5 cut-off; `prob_late` is the underlying probability of "late".

*(Beyond prediction, `ML.WEIGHTS` shows what the model learned and `ML.EXPLAIN_PREDICT` shows why it
made a given call — left here as further exploration.)*

In [ ]:
# Use 1 — score specific flights. Two hypothetical flights: an evening departure from a congested
# hub in peak summer, vs an early-morning short hop in the calm shoulder season.
run_query('''
SELECT
  AIRLINE, ORIGIN_AIRPORT AS origin, DESTINATION_AIRPORT AS dest, dep_hour,
  predicted_is_delayed AS predicted_late,
  ROUND((SELECT prob FROM UNNEST(predicted_is_delayed_probs) WHERE label = 1), 3) AS prob_late
FROM ML.PREDICT(
  MODEL `ccbd-20260603-gpappa.flights_2015.delay_logreg`,
  (
    SELECT 7 AS MONTH, 5 AS DAY_OF_WEEK, 19 AS dep_hour, 'UA' AS AIRLINE,
           'EWR' AS ORIGIN_AIRPORT, 'SFO' AS DESTINATION_AIRPORT, 2565 AS DISTANCE, 360 AS SCHEDULED_TIME
    UNION ALL
    SELECT 9, 2, 6, 'DL', 'ATL', 'MCO', 404, 95
  )
)
''')

In [ ]:
# Use 2 — what-if analysis. Take one flight and vary only the departure hour: the predicted
# late-probability climbs through the day, the model's echo of the Q2 propagation pattern.
run_query('''
SELECT
  dep_hour,
  ROUND((SELECT prob FROM UNNEST(predicted_is_delayed_probs) WHERE label = 1), 3) AS prob_late
FROM ML.PREDICT(
  MODEL `ccbd-20260603-gpappa.flights_2015.delay_logreg`,
  (
    SELECT h AS dep_hour, 7 AS MONTH, 5 AS DAY_OF_WEEK, 'UA' AS AIRLINE,
           'EWR' AS ORIGIN_AIRPORT, 'SFO' AS DESTINATION_AIRPORT, 2565 AS DISTANCE, 360 AS SCHEDULED_TIME
    FROM UNNEST([6, 9, 12, 15, 18, 21]) AS h
  )
)
ORDER BY dep_hour
''')

# Conclusions

The six queries tell one coherent story:

- **Propagation is the core mechanism.** Late-arriving aircraft cause ~40% of delay minutes
  (Q5), which is why delays build through the day (Q2) and why on-time rates split carriers (Q1).
- **Geography and season modulate it.** A handful of hubs (Chicago, NYC) inject congestion
  (Q3); winter brings cancellations, summer brings delays (Q4).
- **Physics offers relief.** Longer flights recover late departures in cruise (Q6), already
  visible as `dep_delay > arr_delay` for most carriers (Q1).
- **Prediction.** A pre-departure logistic model (the ML section) confirms these features carry
  real signal for "will it be late?".

A **Data Studio** dashboard over these results is the separate deliverable.